In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
df = pd.read_csv("/content/Agriculture_price_dataset.csv")

In [3]:
df["Price Date"] = pd.to_datetime(df["Price Date"], errors="coerce")

df["Year"] = df["Price Date"].dt.year
df["Month"] = df["Price Date"].dt.month
df["Day"] = df["Price Date"].dt.day

df.drop(columns=["Price Date"], inplace=True)

In [4]:
X = df.drop("Modal_Price", axis=1)
y = df["Modal_Price"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [6]:
categorical = [
    "STATE",
    "District Name",
    "Market Name",
    "Commodity",
    "Variety",
    "Grade"
]

numeric = [
    "Min_Price",
    "Max_Price",
    "Year",
    "Month",
    "Day"
]

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(handle_unknown="use_encoded_value",
                           unknown_value=-1),
            categorical
        ),
        (
            "num",
            "passthrough",
            numeric
        )
    ]
)

In [8]:
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

In [9]:
model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['STATE', 'District Name',
                                                   'Market Name', 'Commodity',
                                                   'Variety', 'Grade']),
                                                 ('num', 'passthrough',
                                                  ['Min_Price', 'Max_Price',
                                                   'Year', 'Month', 'Day'])])),
                ('rf',
                 RandomForestRegressor(max_depth=15, n_jobs=-1,
                                       random_state=42))])

In [10]:
pred = model.predict(X_test)

In [11]:
from sklearn.metrics import mean_absolute_error

from sklearn.metrics import mean_squared_error

from sklearn.metrics import r2_score

In [12]:
print("MAE:",mean_absolute_error(y_test,pred))

print("RMSE:",np.sqrt(mean_squared_error(y_test,pred)))

print("R2:",r2_score(y_test,pred))

MAE: 41.04788542726512
RMSE: 333.88479089401994
R2: 0.9718783811814256


In [14]:
import joblib
joblib.dump(model,"crop_price_model.pkl")

['crop_price_model.pkl']

In [16]:
from google.colab import files


files.download("crop_price_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>